# American Option Pricing with a Binomial Tree (Continuous Dividend Yield)

This notebook prices **American and European Call and Put options** on **KO (Coca-Cola)** - a stock that pays dividends - with the **CRR binomial tree**, using a **continuous dividend yield $q$**.

**Plan:**
1. fetch KO data + risk-free rate
2. historical volatility from log returns
3. calculate the continuous dividend yield $q$ from KO's real dividend payments
4. tree parameters - only the probability $p$ changes ($r \to r-q$)
5. price American & European options + early-exercise premium
6. validate the European tree prices against the closed-form Black-Scholes-Merton formula
7. implied volatility with Brent's method

## Inputs

| Input | Symbol | Where we get it |
|---|---|---|
| Current stock price | $S_0$ | Market (yfinance, latest close) |
| Strike price | $K$ | Option contract |
| Time to maturity | $T$ | Expiry date - today (in years) |
| Risk-free rate | $r$ | Treasury bill yield (`^IRX`) |
| Volatility | $\sigma$ | Historical log returns of the stock |
| **Dividend yield** | $q$ | KO's actual dividend payments |
| **Tree steps** | $N$ | Chosen by us (more steps = more accurate, slower) |

The tree prices **American** options, and because dividends are paid **continuously** (tiny amounts every instant), the stock price has **no jumps** - unlike the discrete-dividend notebook.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
import scipy.stats as stats
from scipy.optimize import brentq

# Fetch 1 year of historical daily data for KO
ticker_symbol = "KO"
df = yf.Ticker(ticker_symbol).history(period="1y")

# Extract the most recent closing price as the current stock price (S0)
S0 = df['Close'].iloc[-1]
print(f"Current {ticker_symbol} Stock Price (S0): ${S0:.2f}")

Current KO Stock Price (S0): $93.81


## Volatility from historical data

The model needs $\sigma$ = **annualized volatility** of the stock's log returns.

Daily log return:

$$r_t = \ln\!\left(\frac{P_t}{P_{t-1}}\right)$$

Log returns add up over time, so their standard deviation grows with the square root of time. With 252 trading days in a year:

$$\boxed{\sigma_{\text{annual}} = \operatorname{std}(r_t)\times\sqrt{252}}$$

**Assumption:** log returns are (roughly) normally distributed with constant volatility - this is what the binomial tree also assumes when it picks the up/down factors $u$ and $d$.

In [2]:
# Calculate daily log returns: ln(P_t / P_t-1)
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

# Drop missing values if any
log_returns = df['Log_Return'].dropna()

# standard deviation and annualization
daily_volatility = log_returns.std()
historical_volatility = daily_volatility * np.sqrt(252)

print(f"Daily Volatility: {daily_volatility:.6f}")
print(f"Annualized Historical Volatility (sigma): {historical_volatility:.4f} ({historical_volatility * 100:.2f}%)")

Daily Volatility: 0.011935
Annualized Historical Volatility (sigma): 0.1895 (18.95%)


## How to calculate the continuous dividend yield $q$

**In simple words:** a continuous dividend yield $q$ means *"every year the stock pays out q% of its price as dividends, spread evenly over the year"*. Real companies pay 4 big checks a year, but for pricing we **smooth them out into a continuous stream** - the standard trick for stocks and indexes that pay regularly.

### Three ways to calculate $q$ from real data

1. **Trailing yield (what we use below)** - add up the dividends paid in the last 12 months and divide by today's price:

$$q = \frac{\sum \text{dividends paid in the last 12 months}}{S_0}$$

2. **Reported yield** - Yahoo Finance publishes an annual dividend yield (`Ticker.info['dividendYield']`); convert percent to decimal: $q = \text{yield}/100$.

3. **Forward yield** - expected dividends next 12 months divided by $S_0$.

### Why the tree uses $r - q$

Over one step $\Delta t$, the stock grows on average by $(r-q)\Delta t$ in the risk-neutral world: the shareholder receives the dividends, so the **stock itself** grows slower. The dividends themselves are *not* reinvested into the tree - they leave the stock.

In [3]:
# -----------------------------------------------------------------
# Calculate the continuous dividend yield q
# Method 1 (used below): trailing yield = dividends last 12 months / price
# Method 2 (cross-check): yield reported by Yahoo Finance
# -----------------------------------------------------------------
tk = yf.Ticker(ticker_symbol)
div_series = tk.dividends                       # past dividend payments (ex-date, amount)

if len(div_series) > 0:
    if div_series.index.tz is not None:
        div_series.index = div_series.index.tz_localize(None)

    last_12m = div_series[div_series.index >= (datetime.now() - pd.Timedelta(days=365))]
    total_divs_12m = float(last_12m.sum())
    q_trailing = total_divs_12m / S0

    print("Dividend payments in the last 12 months:")
    for date, amt in last_12m.items():
        print(f"   {date.date()}   ${amt:.4f}")
    print(f"\nTotal dividends (12m) : ${total_divs_12m:.4f}")
    print(f"q (trailing) = total / S0 = {total_divs_12m:.4f} / {S0:.2f} = {q_trailing:.4f} ({q_trailing * 100:.2f}%)")
else:
    q_trailing = np.nan
    print(f"WARNING: no dividend history found for {ticker_symbol}")

# Method 2: annual yield reported by Yahoo Finance (cross-check)
try:
    dy = tk.info.get('dividendYield')
    q_reported = (dy / 100) if (dy and dy > 1) else float(dy or np.nan)
except Exception:
    q_reported = np.nan

# use the trailing yield if available, otherwise the reported one
if np.isfinite(q_trailing) and q_trailing > 0:
    q = q_trailing
elif np.isfinite(q_reported) and q_reported > 0:
    q = q_reported
else:
    q = 0.02                                   # last-resort default (2%)

print(f"\nyfinance reported dividend yield : {q_reported * 100:.2f}%")
print(f"Continuous dividend yield used q : {q:.4f} ({q * 100:.2f}%)")

Dividend payments in the last 12 months:
   2025-12-01   $0.5100
   2026-03-13   $0.5300
   2026-06-15   $0.5300
   2026-09-15   $0.5300

Total dividends (12m) : $2.1000
q (trailing) = total / S0 = 2.1000 / 93.81 = 0.0224 (2.24%)



yfinance reported dividend yield : 2.26%
Continuous dividend yield used q : 0.0224 (2.24%)


## Binomial tree with a continuous dividend yield $q$

The tree's **shape does not change**: still $\Delta t = T/N$, $u = e^{\sigma\sqrt{\Delta t}}$, $d = 1/u$. Only the **risk-neutral probability** changes, because the stock's drift becomes $r - q$:

$$p = \frac{e^{(r - q)\,\Delta t} - d}{u - d}$$

Because dividends are paid continuously, the stock has **no jumps**, which gives two nice results:

- the **real stock price** is used directly in every payoff (nothing to add or subtract at nodes)
- **exercising an American call early is never optimal** - you would give up the remaining option time-value just to collect tiny continuous dividends, so **American call = European call**
- **American puts** can still be exercised early (same reason as with no dividends)

## Definition of option parameters

- **Strike** $K$ - nearest-dollar strike (at the money)
- **Time to maturity** $T = \text{days to expiry}/365.25$
- **Risk-free rate** $r$ - from `^IRX`
- **Dividend yield** $q$ - calculated above
- **Volatility** $\sigma$ - from KO's historical log returns
- **Steps** $N = 500$

The cell builds the tree parameters (with the **probability using $r-q$**) and defines `crr_price()`.

In [4]:
# Define option parameters

K = round(S0)                # nearest-dollar strike = at the money
expiry_date = datetime(2027, 6, 18)
today = datetime.now()

# ----------------------------------------------------------------------- #

# time to maturity (T) in years
T = (expiry_date - today).days / 365.25
print(f"Strike Price (K)           : ${K}")
print(f"Time to Maturity (T)       : {T:.8f} years ({(expiry_date - today).days} days)")

# Risk-Free Rate from ^IRX and historical volatility:
irx = yf.Ticker("^IRX")
# recent close price
r = irx.history(period="5d")['Close'].iloc[-1] / 100
sigma = historical_volatility
print(f"Risk-Free Rate (r)         : {r:.4f} ({r * 100:.2f}%)")
print(f"Dividend Yield (q)         : {q:.4f} ({q * 100:.2f}%)")
print(f"Volatility Used (sigma)    : {sigma:.4f} ({sigma * 100:.2f}%)")

# binomial tree parameters (Cox-Ross-Rubinstein, drift r - q)
N = 500                      # number of steps
dt = T / N
u = np.exp(sigma * np.sqrt(dt))
d = 1.0 / u
p = (np.exp((r - q) * dt) - d) / (u - d)
print(f"Tree Steps (N)             : {N}    dt = {dt:.6f}")
print(f"Up (u) / Down (d) / Prob (p): {u:.6f} / {d:.6f} / {p:.4f}")

def crr_price(S, K, T, r, q, sigma, N, option='call', style='american'):
    """
    Price an option with the CRR binomial tree on a stock paying a
    continuous dividend yield q (risk-neutral drift r - q).
    option = 'call' or 'put'
    style  = 'american' or 'european'
    """
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u
    p = (np.exp((r - q) * dt) - d) / (u - d)
    disc = np.exp(-r * dt)

    # option value at expiry (last step): stock prices S*u^(N-j)*d^j
    ST = S * u ** (np.arange(N, -1, -1)) * d ** (np.arange(0, N + 1))
    if option == 'call':
        vals = np.maximum(ST - K, 0.0)
    else:
        vals = np.maximum(K - ST, 0.0)

    # step backwards through the tree
    for i in range(N - 1, -1, -1):
        Si = S * u ** (np.arange(i, -1, -1)) * d ** (np.arange(0, i + 1))
        vals = disc * (p * vals[:-1] + (1 - p) * vals[1:])
        if style == 'american':
            if option == 'call':
                vals = np.maximum(vals, Si - K)
            else:
                vals = np.maximum(vals, K - Si)
    return float(vals[0])

Strike Price (K)           : $94
Time to Maturity (T)       : 0.65982204 years (241 days)


Risk-Free Rate (r)         : 0.0408 (4.08%)
Dividend Yield (q)         : 0.0224 (2.24%)
Volatility Used (sigma)    : 0.1895 (18.95%)
Tree Steps (N)             : 500    dt = 0.001320
Up (u) / Down (d) / Prob (p): 1.006906 / 0.993141 / 0.5000


## Option prices and the early-exercise premium

The cell below prices all four combinations (American/European $\times$ call/put).

**Early-exercise premium:**

$$\text{premium} = \text{American price} - \text{European price} \;\ge\ 0$$

What to expect with **continuous dividends**:
- **American call = European call** - with a continuous yield there is never a good reason to exercise early (the premium is exactly 0 in the tree).
- **American put > European put** - early exercise still adds value, often even more than without dividends (the stock you keep stops paying the yield).

In [5]:
# option prices
call_amer = crr_price(S0, K, T, r, q, sigma, N, 'call', 'american')
put_amer = crr_price(S0, K, T, r, q, sigma, N, 'put', 'american')
call_eur = crr_price(S0, K, T, r, q, sigma, N, 'call', 'european')
put_eur = crr_price(S0, K, T, r, q, sigma, N, 'put', 'european')

print("=" * 45)
print(f"Underlying Asset   : {ticker_symbol}")
print(f"Current Stock Price: ${S0:.2f}")
print(f"Strike Price       : ${K}")
print(f"Expiration Date    : {expiry_date}    T = {T:.4f}")
print(f"Dividend Yield (q) : {q * 100:.2f}%")
print(f"Volatility Used    : {sigma * 100:.2f}%")
print(f"Tree Steps (N)     : {N}")
print("-" * 45)
print(f"American Call      : ${call_amer:.2f}")
print(f"American Put       : ${put_amer:.2f}")
print(f"European Call      : ${call_eur:.2f}")
print(f"European Put       : ${put_eur:.2f}")
print("-" * 45)
print(f"Early-exercise premium  Call: ${call_amer - call_eur:.4f}   (expect 0 with continuous dividends)")
print(f"Early-exercise premium  Put : ${put_amer - put_eur:.4f}   (expect > 0)")
print("=" * 45)

Underlying Asset   : KO
Current Stock Price: $93.81
Strike Price       : $94
Expiration Date    : 2027-06-18 00:00:00    T = 0.6598
Dividend Yield (q) : 2.24%
Volatility Used    : 18.95%
Tree Steps (N)     : 500
---------------------------------------------
American Call      : $6.12
American Put       : $5.31
European Call      : $6.12
European Put       : $5.19
---------------------------------------------
Early-exercise premium  Call: $0.0000   (expect 0 with continuous dividends)
Early-exercise premium  Put : $0.1182   (expect > 0)


## Check: tree vs the closed-form Black-Scholes-Merton formula

For **European** options the tree and the formula must agree (within the tree's small discretization error):

$$C_{\text{tree}} \approx C_{\text{BSM}} = S_0e^{-qT}N(d_1) - Ke^{-rT}N(d_2), \qquad d_1 = \frac{\ln(S_0/K) + (r - q + \tfrac12\sigma^2)T}{\sigma\sqrt{T}}$$

The cell prints both side by side with the difference - at $N = 500$ it should be a fraction of a cent. This is a free cross-check of both implementations.

In [6]:
# Closed-form Black-Scholes-Merton with dividend yield q (validation reference)
def bsm_price(CP, S0, K, T, r, q, sigma):
    d1 = (np.log(S0 / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if CP.upper() == 'CALL':
        return S0 * np.exp(-q * T) * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)
    return K * np.exp(-r * T) * stats.norm.cdf(-d2) - S0 * np.exp(-q * T) * stats.norm.cdf(-d1)

call_bsm = bsm_price('CALL', S0, K, T, r, q, sigma)
put_bsm = bsm_price('PUT', S0, K, T, r, q, sigma)

print(f"{'':24} {'Call':>12} {'Put':>12}")
print(f"{'Tree (European, N=500)':<24} {call_eur:>12.4f} {put_eur:>12.4f}")
print(f"{'Black-Scholes-Merton':<24} {call_bsm:>12.4f} {put_bsm:>12.4f}")
print(f"{'difference':<24} {abs(call_eur - call_bsm):>12.6f} {abs(put_eur - put_bsm):>12.6f}")

                                 Call          Put
Tree (European, N=500)         6.1151       5.1905
Black-Scholes-Merton           6.1150       5.1905
difference                   0.000049     0.000049


## Implied Volatility using Brent's root-finding method

**Normal way:** we know $\sigma$ -> compute the price (here: run the tree). **Implied volatility:** we know the *market price* -> find the $\sigma$ that makes the tree produce it:

$$\text{find } \sigma \text{ such that } \text{Tree}_q(S_0, K, T, r, q, \sigma) = \text{market price}$$

Same Brent root-finder as before - the pricer inside is the binomial tree **with the dividend yield**, so the dividend effect is not mistaken for volatility. We search American prices, because listed stock options are American.

In [7]:
def tree_price(CP, S0, K, sigma, T, r, q):
    """CRR binomial price with dividend yield q (American style), used by the root-finder."""
    if sigma <= 0:
        return 0.0
    option = 'call' if CP.upper() == 'CALL' else 'put'
    return crr_price(S0, K, T, r, q, sigma, N, option, 'american')

def implied_volatility(market_price, CP, S0, K, T, r, q):
    """
    Finds the implied volatility by reverse-engineering the binomial tree price
    using Brent's root-finding method.
    """
    # objective function: tree price - market price
    def objective_function(sigma):
        return tree_price(CP, S0, K, sigma, T, r, q) - market_price

    try:
        # search for volatility between 1% and 500%
        iv = brentq(objective_function, 1e-2, 5.0)
        return iv
    except ValueError:
        # returns NaN if the market price is invalid
        return np.nan

if __name__ == "__main__":
    market_price_call = 4.50    # observed market prices
    market_price_put = 3.50

    iv_call = implied_volatility(market_price_call, 'CALL', S0, K, T, r, q)
    iv_put = implied_volatility(market_price_put, 'PUT', S0, K, T, r, q)

    print()
    print(f"Market prices used : call ${market_price_call:.2f}, put ${market_price_put:.2f}")
    print()
    print(f"Implied Volatility_CALL: {iv_call:.4f} ({iv_call * 100:.2f}%)")
    print()
    print(f"Implied Volatility_PUT: {iv_put:.4f} ({iv_put * 100:.2f}%)")


Market prices used : call $4.50, put $3.50

Implied Volatility_CALL: 0.1350 (13.50%)

Implied Volatility_PUT: 0.1285 (12.85%)
